In [1]:
!pip install xgboost
!pip install scikit-learn
!pip install pandas
!pip install numpy
!pip install joblib

In [2]:
!python train_models.py

python3: can't open file '/content/train_models.py': [Errno 2] No such file or directory


In [ ]:
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    accuracy_score,
    roc_auc_score
)

from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

DATASET_PATH = r"/content/sample_data/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv"

MODEL_DIR = "models"
DATA_DIR = "data"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

print("Loading dataset...")

df = pd.read_csv(DATASET_PATH)

print("Shape:", df.shape)

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

print("Columns:")
print(df.columns.tolist())

# ============================================================
# DATETIME PROCESSING
# ============================================================

datetime_cols = [
    "start_datetime",
    "end_datetime",
    "created_date",
    "modified_datetime",
    "closed_datetime",
    "resolved_datetime"
]

for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

# ============================================================
# BOOLEANS
# ============================================================

bool_cols = [
    "requires_road_closure",
    "authenticated"
]

for col in bool_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .map({
                "true": 1,
                "false": 0,
                "yes": 1,
                "no": 0
            })
        )

# ============================================================
# NUMERIC CONVERSION
# ============================================================

numeric_cols = [
    "latitude",
    "longitude",
    "endlatitude",
    "endlongitude",
    "age_of_truck"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

df = df.drop_duplicates()

# ============================================================
# DURATION
# ============================================================

print("Creating duration feature...")

df["duration_minutes"] = (
    (
        df["end_datetime"] -
        df["start_datetime"]
    )
    .dt.total_seconds()
    / 60
)

df["duration_minutes"] = (
    df["duration_minutes"]
    .clip(lower=0)
)

df["duration_minutes"] = (
    df["duration_minutes"]
    .fillna(0)
)

df["duration_minutes"] = (
    df["duration_minutes"]
    .clip(upper=1440)
)

event_duration = (
    df[df["duration_minutes"] > 0]
    .groupby("event_type")
    ["duration_minutes"]
    .median()
)

def fill_duration(row):

    if row["duration_minutes"] > 0:
        return row["duration_minutes"]

    return event_duration.get(
        row["event_type"],
        30
    )

df["duration_minutes"] = (
    df.apply(fill_duration, axis=1)
)

# ============================================================
# TIME FEATURES
# ============================================================

df["hour"] = (
    df["start_datetime"]
    .dt.hour
)

df["weekday"] = (
    df["start_datetime"]
    .dt.weekday
)

df["month"] = (
    df["start_datetime"]
    .dt.month
)

df["weekend"] = (
    df["weekday"]
    .isin([5, 6])
).astype(int)

df["peak_hour"] = (
    df["hour"]
    .isin([8, 9, 10, 17, 18, 19])
).astype(int)

# ============================================================
# RISK FEATURES
# ============================================================

print("Building risk tables...")

event_freq = (
    df["event_type"]
    .value_counts(normalize=True)
)

df["event_frequency_score"] = (
    df["event_type"]
    .map(event_freq)
)

closure_risk = (
    df.groupby("event_type")
    ["requires_road_closure"]
    .mean()
)

df["closure_risk"] = (
    df["event_type"]
    .map(closure_risk)
)

duration_risk = (
    df.groupby("event_type")
    ["duration_minutes"]
    .median()
)

df["duration_risk"] = (
    df["event_type"]
    .map(duration_risk)
)

# ============================================================
# JUNCTION RISK
# ============================================================

junction_risk = {}

for j in df["junction"].dropna().unique():

    subset = df[df["junction"] == j]

    score = (
        0.5 * len(subset)
        +
        0.5 * subset["duration_minutes"].mean()
    )

    junction_risk[j] = float(score)

# ============================================================
# CORRIDOR RISK
# ============================================================

corridor_risk = {}

for c in df["corridor"].dropna().unique():

    subset = df[df["corridor"] == c]

    score = (
        0.5 * len(subset)
        +
        0.5 * subset["duration_minutes"].mean()
    )

    corridor_risk[c] = float(score)

# ============================================================
# HISTORICAL IMPACT
# ============================================================

historical = (
    df.groupby(
        [
            "event_type",
            "corridor",
            "hour"
        ]
    )
    ["duration_minutes"]
    .median()
)

df["historical_impact"] = (
    df.set_index(
        [
            "event_type",
            "corridor",
            "hour"
        ]
    )
    .index
    .map(historical)
)

junction_count = (
    df["junction"]
    .value_counts()
)

junction_duration = (
    df.groupby("junction")
    ["duration_minutes"]
    .mean()
)

corridor_count = (
    df["corridor"]
    .value_counts()
)

corridor_duration = (
    df.groupby("corridor")
    ["duration_minutes"]
    .mean()
)

df["junction_count"] = (
    df["junction"]
    .map(junction_count)
)

df["junction_duration"] = (
    df["junction"]
    .map(junction_duration)
)

df["corridor_count"] = (
    df["corridor"]
    .map(corridor_count)
)

df["corridor_duration"] = (
    df["corridor"]
    .map(corridor_duration)
)

risk_cols = [
    "event_frequency_score",
    "closure_risk",
    "duration_risk",
    "junction_count",
    "junction_duration",
    "corridor_count",
    "corridor_duration",
    "historical_impact"
]

for col in risk_cols:
    df[col] = (
        df[col]
        .fillna(df[col].median())
    )

# ============================================================
# TRAFFIC IMPACT INDEX
# ============================================================

scaler = MinMaxScaler()

df[risk_cols] = scaler.fit_transform(
    df[risk_cols]
)

tii_scaler = MinMaxScaler(
    feature_range=(0, 100)
)

df["traffic_impact_index"] = (
    tii_scaler.fit_transform(
        df[risk_cols]
        .mean(axis=1)
        .values.reshape(-1, 1)
    )
)

# ============================================================
# FEATURES
# ============================================================

features = [

    "event_type",
    "event_cause",
    "veh_type",
    "corridor",
    "zone",
    "junction",

    "latitude",
    "longitude",

    "hour",
    "weekday",
    "month",

    "weekend",
    "peak_hour",

    "event_frequency_score",

    "closure_risk",
    "duration_risk",

    "junction_count",
    "junction_duration",

    "corridor_count",
    "corridor_duration",

    "historical_impact"
]

# Keep only existing columns

features = [
    c for c in features
    if c in df.columns
]

# ============================================================
# FILL MISSING
# ============================================================

for col in features:

    if df[col].dtype == object:
        df[col] = (
            df[col]
            .fillna("Unknown")
        )
    else:
        df[col] = (
            df[col]
            .fillna(df[col].median())
        )

# ============================================================
# CATEGORICAL
# ============================================================

cat_cols = [
    c for c in [
        "event_type",
        "event_cause",
        "veh_type",
        "corridor",
        "zone",
        "junction"
    ]
    if c in features
]

num_cols = [
    c for c in features
    if c not in cat_cols
]

preprocessor = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            cat_cols
        )
    ],
    remainder="passthrough"
)

# ============================================================
# DURATION MODEL
# ============================================================

print("\nTraining duration model...")

X = df[features]
y = df["duration_minutes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

duration_model = Pipeline(
    [
        ("prep", preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                random_state=42
            )
        )
    ]
)

duration_model.fit(
    X_train,
    y_train
)

pred = duration_model.predict(X_test)

print(
    "Duration R2:",
    round(r2_score(y_test, pred), 4)
)

print(
    "Duration MAE:",
    round(mean_absolute_error(y_test, pred), 4)
)

# ============================================================
# CLOSURE MODEL
# ============================================================

print("\nTraining closure model...")

if "requires_road_closure" in df.columns:

    y = (
        df["requires_road_closure"]
        .fillna(0)
        .astype(int)
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    closure_model = Pipeline(
        [
            ("prep", preprocessor),
            (
                "model",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=6,
                    learning_rate=0.05,
                    random_state=42
                )
            )
        ]
    )

    closure_model.fit(
        X_train,
        y_train
    )

    prob = closure_model.predict_proba(X_test)[:, 1]

    pred = (
        prob > 0.5
    ).astype(int)

    print(
        "Closure Accuracy:",
        round(
            accuracy_score(
                y_test,
                pred
            ),
            4
        )
    )

    print(
        "Closure ROC AUC:",
        round(
            roc_auc_score(
                y_test,
                prob
            ),
            4
        )
    )

else:

    closure_model = None

# ============================================================
# TII MODEL
# ============================================================

print("\nTraining TII model...")

y = df["traffic_impact_index"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

tii_model = Pipeline(
    [
        ("prep", preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                random_state=42
            )
        )
    ]
)

tii_model.fit(
    X_train,
    y_train
)

pred = tii_model.predict(X_test)

print(
    "TII R2:",
    round(
        r2_score(
            y_test,
            pred
        ),
        4
    )
)

# ============================================================
# SAVE MODELS
# ============================================================

print("\nSaving models...")

joblib.dump(
    duration_model,
    os.path.join(
        MODEL_DIR,
        "duration_model.pkl"
    )
)

joblib.dump(
    closure_model,
    os.path.join(
        MODEL_DIR,
        "closure_model.pkl"
    )
)

joblib.dump(
    tii_model,
    os.path.join(
        MODEL_DIR,
        "tii_model.pkl"
    )
)

joblib.dump(
    features,
    os.path.join(
        MODEL_DIR,
        "feature_columns.pkl"
    )
)

joblib.dump(
    scaler,
    os.path.join(
        MODEL_DIR,
        "risk_scaler.pkl"
    )
)

# ============================================================
# SAVE JSONS
# ============================================================

event_severity = {}

for e in df["event_type"].unique():

    subset = df[
        df["event_type"] == e
    ]

    event_severity[e] = float(
        subset[
            "traffic_impact_index"
        ].mean()
    )

with open(
    os.path.join(
        DATA_DIR,
        "corridor_risk.json"
    ),
    "w"
) as f:
    json.dump(
        corridor_risk,
        f,
        indent=2
    )

with open(
    os.path.join(
        DATA_DIR,
        "junction_risk.json"
    ),
    "w"
) as f:
    json.dump(
        junction_risk,
        f,
        indent=2
    )

with open(
    os.path.join(
        DATA_DIR,
        "event_severity.json"
    ),
    "w"
) as f:
    json.dump(
        event_severity,
        f,
        indent=2
    )

print("\nSUCCESS")
print("Models saved in models/")
print("Risk tables saved in data/")

Loading dataset...
Shape: (8173, 46)
Columns:
['id', 'event_type', 'latitude', 'longitude', 'endlatitude', 'endlongitude', 'address', 'end_address', 'event_cause', 'requires_road_closure', 'start_datetime', 'end_datetime', 'status', 'authenticated', 'modified_datetime', 'map_file', 'direction', 'description', 'veh_type', 'veh_no', 'corridor', 'priority', 'cargo_material', 'reason_breakdown', 'age_of_truck', 'created_date', 'route_path', 'client_id', 'created_by_id', 'last_modified_by_id', 'assigned_to_police_id', 'citizen_accident_id', 'comment', 'police_station', 'meta_data', 'kgid', 'resolved_at_address', 'resolved_at_latitude', 'resolved_at_longitude', 'closed_by_id', 'closed_datetime', 'resolved_by_id', 'resolved_datetime', 'gba_identifier', 'zone', 'junction']
Creating duration feature...
Building risk tables...

Training duration model...
Duration R2: 0.7753
Duration MAE: 17.3043

Training closure model...
Closure Accuracy: 0.9235
Closure ROC AUC: 0.7953

Training TII model...
TI

In [ ]:
from google.colab import files

files.download("models/duration_model.pkl")
files.download("models/closure_model.pkl")
files.download("models/tii_model.pkl")
files.download("models/feature_columns.pkl")
files.download("models/risk_scaler.pkl")

files.download("data/corridor_risk.json")
files.download("data/junction_risk.json")
files.download("data/event_severity.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip show scikit-learn
!pip show xgboost
!pip show pandas
!pip show numpy
!pip show joblib

In [ ]:
import sys
import platform

print("Python Version:", sys.version)
print("Platform:", platform.platform())

packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "xgboost",
    "joblib",
    "scipy",
    "matplotlib",
    "seaborn"
]

for pkg in packages:
    try:
        module = __import__(pkg.replace("-", "_"))
        print(f"{pkg}: {module.__version__}")
    except Exception as e:
        print(f"{pkg}: Not Installed")

In [ ]:
print(df["duration_minutes"].describe())

print("\nMAX:")
print(df["duration_minutes"].max())

print("\nTOP 20:")
print(df["duration_minutes"].sort_values(ascending=False).head(20))